# Analytic partial derivatives of the Bolin sensitivity Be

`buffderiv()` returns, in closed form,

$$\frac{\partial B_e}{\partial T},\qquad
  \frac{\partial B_e}{\partial S},\qquad
  \frac{\partial B_e}{\partial C_T},\qquad
  \frac{\partial B_e}{\partial A_T}$$

with $B_e = \partial C_T / \partial [\mathrm{CO_2^*}]$ at constant $A_T, T, S$.

Every equilibrium constant is taken from seacarb's own `K1()`, `K2()`, `Kb()`, `Kw()`,
`Ksi()`, `K1p()`, `K2p()`, `K3p()`, `Ks()`, `Kf()`, `kconv()` and `bor()`, fetched exactly
as `calculate_carb()` fetches them. Nothing is transcribed. The only thing seacarb does not
supply is $dK/dT$ and $dK/dS$; those live in `dlnK.R` as logarithmic derivatives.

Silicate is **monoprotic (K1si only)**, as in `carb()`, which calls `calculate_carb()` with
`fullresult=FALSE` and therefore never forms K2si.

### Two seacarb traps this function guards against

Both were verified in the seacarb 3.3.4 source, and neither emits a warning.

1. `k1k2="x"` does `is_outrange <- T>35 | T<2 | S<19 | S>43`, then silently switches to
   Waters et al. (2014). That is exactly where the polar CMIP6 surface cells live.
2. `kf="x"` silently switches between Perez & Fraga and Dickson & Riley by T,S range,
   which changes `kSWS2total` and hence Kw, Ksi, K1p, K2p and K3p by 0.5 to 0.9 %.

`buffderiv()` stops unless you pass `k1k2="l"` and `kf="dg"` explicitly.

In [ ]:
source("setup.R")   # library(seacarb) + the buffer functions from ../R
packageVersion("seacarb")

## 1. A single point

In [ ]:
buffderiv(flag = 15, var1 = 2300e-6, var2 = 2000e-6,
          S = 35, T = 20, Pt = 0.2e-6, Sit = 3e-6,
          k1k2 = "l", kf = "dg", ks = "d", pHscale = "T", b = "u74")

## 2. Vectorised over a field

`buffderiv()` is fully vectorised. The single `carb()` call is the only nonlinear solve;
everything after it is algebra.

In [ ]:
n   <- 45000                       # ~ unmasked surface cells of a 360x180 grid
S   <- runif(n, 30, 37)
T   <- runif(n, -1.8, 30)
AT  <- runif(n, 2100, 2400) * 1e-6
CT  <- AT * runif(n, 0.85, 0.97)
Pt  <- runif(n, 0, 2.5) * 1e-6
Sit <- runif(n, 0, 120) * 1e-6

system.time(
  d <- buffderiv(15, AT, CT, S = S, T = T, Pt = Pt, Sit = Sit,
                 k1k2 = "l", kf = "dg", ks = "d", pHscale = "T",
                 b = "u74", warn = "n")
)

In [ ]:
summary(d[, c("Be", "dBe_dT", "dBe_dS", "dBe_dCT", "dBe_dAT")])

## 3. Validation

### (a) The alkalinity residual

`buffderiv()` returns `alk_residual` = Ac(h) + Anc(h) - ALK. If our Anc were not exactly
the alkalinity that `carb()` itself solves, we would be differentiating the wrong function.
A structural error, for instance diprotic instead of monoprotic silicate at
Sit = 60 umol/kg, would show up here at the 1e-6 level.

In [ ]:
max(abs(d$alk_residual) / AT)      # expect ~1e-16

### (b) Finite differences

Each analytic derivative is checked against an independent Richardson-extrapolated central
difference taken through the full nonlinear `carb()` solve.

In [ ]:
rich <- function(f, x, eps) {
  d1 <- (f(x + eps)     - f(x - eps))     / (2 * eps)
  d2 <- (f(x + 2 * eps) - f(x - 2 * eps)) / (4 * eps)
  (4 * d1 - d2) / 3
}

Be_of <- function(S, T, CT, AT, Pt, Sit)
  buffderiv(15, AT, CT, S = S, T = T, Pt = Pt, Sit = Sit,
            k1k2 = "l", kf = "dg", ks = "d", pHscale = "T",
            b = "u74", warn = "n")$Be

In [ ]:
S0 <- 35; T0 <- 20; CT0 <- 2000e-6; AT0 <- 2300e-6
P0 <- 0.2e-6; Si0 <- 3e-6

a <- buffderiv(15, AT0, CT0, S = S0, T = T0, Pt = P0, Sit = Si0,
               k1k2 = "l", kf = "dg", ks = "d", pHscale = "T",
               b = "u74", warn = "n")

num <- c(
  dBe_dT  = rich(function(x) Be_of(S0, x, CT0, AT0, P0, Si0), T0,  1e-3),
  dBe_dS  = rich(function(x) Be_of(x, T0, CT0, AT0, P0, Si0), S0,  1e-3),
  dBe_dCT = rich(function(x) Be_of(S0, T0, x, AT0, P0, Si0), CT0, CT0 * 1e-5),
  dBe_dAT = rich(function(x) Be_of(S0, T0, CT0, x, P0, Si0), AT0, AT0 * 1e-5))

data.frame(analytic    = unlist(a[names(num)]),
           finite_diff = num,
           pct_diff    = 100 * (unlist(a[names(num)]) - num) / num)

### (c) The full test suite

`validate_buffderiv.R` and `sweep.R` give:

| check | worst error |
|---|---|
| `dlnK.R` dK/dT, dK/dS vs FD of seacarb's own K functions | 9.6e-7 % |
| alkalinity residual at `carb()`'s **unpolished** h | 9.7e-11 |
| alkalinity residual after polish, \|resid\|/AT | 2.0e-16 |
| Be vs `buffsun()` at Pt = Sit = 0 | 1.8e-8 % |
| dBe/dY vs FD through `carb()`, 310 points, both solver paths | **1.1e-6 %** |
| dBe/dY vs independent automatic differentiation | **1.0e-11 %** |

The sweep covers T in [-1.8, 32], S in [28, 38], AT in [2000, 2450], CT/AT in [0.83, 0.99],
Pt in [0, 2.5] and Sit in [0, 120] umol/kg. All 300 random points passed.

That 1.1e-6 % is the double-precision finite-difference noise floor, not analytic error. The
independent AD implementation (`ad_dual.R`, `ad_be.R`) agrees to 1.0e-11 %.

### A note on the Newton polish

**Correcting an earlier claim in this notebook.** An earlier version said that `carb()`
converges h only to ~1e-8, and that the Newton polish was what fixed the derivatives. Both
statements were wrong, and the second one was a misdiagnosis worth recording.

The real story: an alkalinity residual of 5.2e-7 was showing up at `carb()`'s h, and it was
tempting to blame SolveSAPHE's convergence tolerance. It was not that. Tightening
SolveSAPHE's `pp_rdel_ah_target` from 1e-8 to 1e-15 changed the returned h by **exactly
zero**, which proves `carb()`'s h was already converged. The residual was caused by our
alkalinity **missing the -[HF] term**, which SolveSAPHE carries explicitly. Adding it dropped
the residual at `carb()`'s *unpolished* h from 5.2e-7 to 9.7e-11.

So what is the polish for? Self-consistency of the implicit function theorem, not repair of
`carb()`. dBe/dY = G_Y - G_h F_Y / F_h is exact only at a point where F(h) = 0. The polish
puts h there to machine precision. It buys the last few digits; it does not fix a bug.

**The diagnostic that matters.** `alk_residual` must be evaluated at `npolish = 0`. With the
polish on it is driven to zero by construction and tells you nothing. That is precisely how
the fluoride bug hid.

`buffsun()$Be` uses `carb()`'s h without polishing, which is why it agrees with `buffderiv()`
to 1.8e-8 % rather than exactly. That residual is `carb()`'s ~1e-10 h tolerance, nothing more.